# BCH Simulator (complete)
این نوت‌بوک شامل پیاده‌سازی کامل شبیه‌ساز BCH (باینری) است — قابل اجرا در Jupyter.
- ساخت میدان GF(2^m)
- محاسبهٔ چندجمله‌ای‌های مینیمال و چندجمله‌ای مولد
- انکُدینگ سیستماتیک
- دِکُدینگ با Berlekamp–Massey، Chien و Forney (برای کدهای دودویی اصلاح با چرخش بیت)
- شبیه‌سازی کانال BSC و محاسبات FER/BER

(نسخهٔ آموزشی — برای mهای خیلی بزرگ نیاز به بهینه‌سازی دارد.)


In [ ]:

# -*- coding: utf-8 -*-
# شبیه‌ساز کامل BCH (باینری) — Jupyter-compatible Python code cell
# این سلول شامل کد کامل شبیه‌ساز است.

# وابستگی‌ها
import numpy as np
import random
from typing import List, Tuple

# GF(2^m) implementation
class GF2m:
    def __init__(self, m: int, prim_poly: int):
        assert m >= 1
        self.m = m
        self.prim = prim_poly
        self.n = (1 << m) - 1
        self.exp = [0] * (2 * self.n)
        self.log = [-1] * (self.n + 1)
        self._build_tables()

    def _build_tables(self):
        x = 1
        for i in range(self.n):
            self.exp[i] = x
            self.log[x] = i
            x <<= 1
            if x & (1 << self.m):
                x ^= self.prim
        for i in range(self.n, 2 * self.n):
            self.exp[i] = self.exp[i - self.n]

    def add(self, a: int, b: int) -> int:
        return a ^ b

    def mul(self, a: int, b: int) -> int:
        if a == 0 or b == 0:
            return 0
        return self.exp[self.log[a] + self.log[b]]

    def inv(self, a: int) -> int:
        if a == 0:
            raise ZeroDivisionError("inverse of 0")
        return self.exp[self.n - self.log[a]]

    def pow(self, a: int, e: int) -> int:
        if a == 0:
            return 0
        return self.exp[(self.log[a] * e) % self.n]

    def element_to_bits(self, a: int) -> List[int]:
        return [(a >> (self.m - 1 - i)) & 1 for i in range(self.m)]

    def bits_to_element(self, bits: List[int]) -> int:
        v = 0
        for b in bits:
            v = (v << 1) | (b & 1)
        return v

# Polynomial helpers (binary)
def trim_poly(p: List[int]) -> List[int]:
    while len(p) > 1 and p[0] == 0:
        p.pop(0)
    return p

def poly_mul_bin(a: List[int], b: List[int]) -> List[int]:
    res = [0] * (len(a) + len(b) - 1)
    for i, ai in enumerate(a):
        if ai:
            for j, bj in enumerate(b):
                res[i + j] ^= bj
    return trim_poly(res)

def poly_add_bin(a: List[int], b: List[int]) -> List[int]:
    la = len(a); lb = len(b)
    if la < lb:
        a = [0] * (lb - la) + a
    if lb < la:
        b = [0] * (la - lb) + b
    return trim_poly([ (ai ^ bi) for ai, bi in zip(a, b) ])

def poly_divmod_bin(dividend: List[int], divisor: List[int]) -> Tuple[List[int], List[int]]:
    A = dividend.copy()
    A = trim_poly(A)
    D = trim_poly(divisor)
    if D == [0]:
        raise ZeroDivisionError("polynomial division by zero")
    n = len(D)
    q = [0] * max(len(A) - n + 1, 1)
    while len(A) >= n:
        if A[0] == 1:
            qpos = len(A) - n
            q[qpos] = 1
            for i in range(n):
                A[i] ^= D[i]
        A.pop(0)
        A = trim_poly(A)
        if not A:
            A = [0]
    return trim_poly(q), trim_poly(A)

# Minimal polynomial via conjugacy class and field-coefficient multiplication
def minimal_polynomial(alpha_pow: int, gf: GF2m) -> List[int]:
    n = gf.n
    conj = []
    seen = set()
    x = alpha_pow % n
    while x not in seen:
        seen.add(x)
        conj.append(x)
        x = (x * 2) % n
    poly_field = [1]
    for e in conj:
        root = gf.exp[e]
        new = [0] * (len(poly_field) + 1)
        for i, coeff in enumerate(poly_field):
            new[i] ^= gf.mul(coeff, root) if coeff != 0 and root != 0 else (coeff & root)
            new[i + 1] ^= coeff
        poly_field = new
    poly_field_desc = poly_field[::-1]
    bin_poly = []
    for a in poly_field_desc:
        if a == 0:
            bin_poly.append(0)
        elif a == 1:
            bin_poly.append(1)
        else:
            bits = gf.element_to_bits(a)
            if bits == [0] * (gf.m - 1) + [1] or bits == [1]:
                bin_poly.append(1)
            else:
                raise ValueError(f"Coefficient {a} of minimal polynomial is not in GF(2)")
    return trim_poly(bin_poly)

def bch_generator_poly(m: int, t: int, prim_poly: int) -> List[int]:
    gf = GF2m(m, prim_poly)
    n = gf.n
    polys = []
    used = set()
    for i in range(1, 2 * t + 1):
        if i in used:
            continue
        cls = []
        x = i % n
        while x not in cls:
            cls.append(x)
            x = (x * 2) % n
        for e in cls:
            used.add(e)
        p = minimal_polynomial(i, gf)
        polys.append(p)
    g = [1]
    for p in polys:
        g = poly_mul_bin(g, p)
    return trim_poly(g)

# Systematic encoder
def systematic_bch_encode(msg_bits: List[int], g: List[int]) -> List[int]:
    k = len(msg_bits)
    deg_g = len(g) - 1
    padded = msg_bits + [0] * deg_g
    _, remainder = poly_divmod_bin(padded, g)
    rem = remainder
    if len(rem) < deg_g:
        rem = [0] * (deg_g - len(rem)) + rem
    codeword = msg_bits + rem
    return codeword

# Syndromes
def compute_syndromes(codeword: List[int], t: int, gf: GF2m) -> List[int]:
    n = gf.n
    synd = []
    for j in range(1, 2 * t + 1):
        a = gf.exp[j]
        s = 0
        for coef in codeword:
            s = gf.mul(s, a) ^ coef
        synd.append(s)
    return synd

# Berlekamp-Massey over GF(2^m)
def berlekamp_massey(synd: List[int], gf: GF2m) -> List[int]:
    N = len(synd)
    C = [1] + [0] * N
    B = [1] + [0] * N
    L = 0
    m = 1
    b = 1
    for n in range(N):
        d = synd[n]
        for i in range(1, L + 1):
            if C[i] != 0 and (n - i) >= 0:
                d = d ^ gf.mul(C[i], synd[n - i])
        if d == 0:
            m += 1
        else:
            T = C.copy()
            coef = gf.mul(d, gf.inv(b))
            for i in range(len(B)):
                if B[i] != 0:
                    C[i + m] ^= gf.mul(coef, B[i])
            if 2 * L <= n:
                L_new = n + 1 - L
                B = T
                b = d
                L = L_new
                m = 1
            else:
                m += 1
    C = C[:L + 1]
    return C

# Chien search and correct
def chien_search_and_correct(codeword: List[int], sigma: List[int], gf: GF2m) -> Tuple[List[int], List[int]]:
    n = gf.n
    cw = codeword.copy()
    error_positions = []
    for i in range(n):
        x = gf.exp[(n - i) % n]
        val = 0
        xp = 1
        for coeff in sigma:
            if coeff != 0:
                val ^= gf.mul(coeff, xp)
            xp = gf.mul(xp, x)
        if val == 0:
            idx = len(cw) - 1 - i
            if 0 <= idx < len(cw):
                cw[idx] ^= 1
                error_positions.append(idx)
    return cw, error_positions

def bch_decode(received: List[int], t: int, gf: GF2m) -> Tuple[List[int], bool, List[int]]:
    synd = compute_syndromes(received, t, gf)
    if all(s == 0 for s in synd):
        return received, True, []
    sigma = berlekamp_massey(synd, gf)
    corrected, errs = chien_search_and_correct(received, sigma, gf)
    synd2 = compute_syndromes(corrected, t, gf)
    success = all(s == 0 for s in synd2)
    return corrected, success, errs

# Channel simulator
def flip_bits(codeword: List[int], p: float) -> Tuple[List[int], List[int]]:
    n = len(codeword)
    rx = codeword.copy()
    errs = []
    for i in range(n):
        if random.random() < p:
            rx[i] ^= 1
            errs.append(i)
    return rx, errs

def simulate_bch(m: int, t: int, prim_poly: int, trials: int = 1000, p_bit: float = 0.01, seed: int = None):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    gf = GF2m(m, prim_poly)
    n = gf.n
    g = bch_generator_poly(m, t, prim_poly)
    k = n - (len(g) - 1)
    total_frame_err = 0
    total_bit_err = 0
    total_bits = trials * k
    for _ in range(trials):
        msg = [random.randint(0, 1) for _ in range(k)]
        cw = systematic_bch_encode(msg, g)
        rx, true_errs = flip_bits(cw, p_bit)
        decoded, success, found_errs = bch_decode(rx, t, gf)
        dec_msg = decoded[:k]
        if dec_msg != msg:
            total_frame_err += 1
            for i in range(k):
                if dec_msg[i] != msg[i]:
                    total_bit_err += 1
    fer = total_frame_err / trials
    ber = total_bit_err / total_bits
    return {
        'm': m, 'n': n, 't': t, 'k': k, 'g': g,
        'trials': trials, 'p_bit': p_bit,
        'FER': fer, 'BER': ber
    }

# Example run
if __name__ == '__main__':
    m = 4
    prim_poly = 0b10011
    t = 2
    print(f"GF(2^{m}) with n = { (1<<m) - 1 }")
    g = bch_generator_poly(m, t, prim_poly)
    print("Generator polynomial g(x) (binary coeffs MSB..LSB):", g)
    k = ( (1<<m) - 1 ) - (len(g) - 1)
    print(f"Estimated k = {k} (information bits)")

    msg = [random.randint(0,1) for _ in range(k)]
    cw = systematic_bch_encode(msg, g)
    print("Message:", msg)
    print("Codeword:", cw)

    err_pos = random.sample(range(len(cw)), t)
    rx = cw.copy()
    for p in err_pos:
        rx[p] ^= 1
    print("Introduced errors at positions:", err_pos)

    decoded, success, found_errs = bch_decode(rx, t, GF2m(m, prim_poly))
    print("Decoding success:", success)
    print("Found error positions (indexes in codeword MSB..LSB):", found_errs)
    if success:
        print("Decoded message equals original?", decoded[:k] == msg)

    res = simulate_bch(m, t, prim_poly, trials=500, p_bit=0.02, seed=42)
    print("Simulation summary:")
    for k2,v in res.items():
        print(k2, ":", v)
